# inkalign: depth/orientation sweep on a free Colab GPU

Runs the full `inkalign` pipeline on a real Vesuvius Challenge segment:

1. stream a small ROI of a surface volume from the open S3 bucket (no credentials),
2. download an official `ink_9um` checkpoint from Hugging Face,
3. sweep ink-detection inference over depth windows and both orientations,
4. score each prediction for text-likeness and report the best window + facing.

**Runtime:** use a GPU runtime (`Runtime > Change runtime type > T4 GPU`).
The ROI is ~16 MB, so the whole sweep fits comfortably in the free tier.

In [ ]:
# 1. Environment — two interpreters, cleanly separated:
#    * inkalign installs into Colab's default Python (it needs zarr 2.x),
#      plus imagecodecs so it can read back infer.py's LZW-compressed TIFFs;
#    * villa's `vesuvius` requires Python 3.14 exactly, so uv manages a 3.14
#      venv for it. The extra packages on that line are deps infer.py needs
#      but the lean vesuvius install does not declare (tifffile, imagecodecs,
#      zarr, nest-asyncio, aiohttp, requests, pyyaml, einops, timm).
#      torch+torchvision come matched from the cu126 index, whose wheels keep
#      sm_60 (P100) kernels that the default cu130 build drops.
# inkalign's sweep bridges the two via --infer-template.
# TORCHDYNAMO_DISABLE: torch.compile needs Triton (CUDA capability >= 7.0);
# eager mode runs on any GPU the free tier assigns, including P100.
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

%pip install -q "git+https://github.com/hilalitvak/inkalign" uv imagecodecs
!uv venv /content/venv --python 3.14
!uv pip install -p /content/venv/bin/python torch==2.13.0 torchvision \
    --index-url https://download.pytorch.org/whl/cu126
!uv pip install -p /content/venv/bin/python \
    "git+https://github.com/ScrollPrize/villa#subdirectory=vesuvius" \
    numba tifffile imagecodecs zarr nest-asyncio aiohttp requests pyyaml \
    einops timm

VENV_PY = "/content/venv/bin/python"
!{VENV_PY} -c "import torch; print('CUDA available:', torch.cuda.is_available())"
# fail fast if the lean install is missing something:
!{VENV_PY} -m vesuvius.ink_detection.inference.infer --help > /dev/null && echo "infer entry point OK"

In [ ]:
# 2. Extract a small full-depth ROI from the real w025 segment (PHerc0139)
VOLUME = ("https://vesuvius-challenge-open-data.s3.amazonaws.com/"
          "PHerc0139/segments/20250108000000-w025_2025010863/surface-volumes/"
          "9.362um-1.2m-113keV-volume-20250728140407.zarr")
!inkalign extract-roi "{VOLUME}" --size 768 --out roi.zarr

# sanity: CPU depth profile of the ROI (what does the CT signal say?)
!inkalign profile roi.zarr --outdir profile_out
from IPython.display import Image, display
display(Image("profile_out/depth_profile.png"))

In [ ]:
# 3. Official cross-scroll ink checkpoint from Hugging Face
%pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download("scrollprize/ink_9um",
                       "hybrid_3d2d-seed42/step-075000.pth")
print(ckpt)

In [ ]:
# 4. The sweep: depth windows x both orientations, scored for text-likeness.
# The infer template points at the 3.14 venv's interpreter, where vesuvius +
# torch live; inkalign itself runs in Colab's Python. (subprocess rather than
# `!` because IPython would expand the template's {placeholders} itself.)
import subprocess, sys
TEMPLATE = (f"{VENV_PY} -m vesuvius.ink_detection.inference.infer "
            "{volume} {checkpoint} {output} "
            "--layer-start {layer_start} --layer-end {layer_end} --direction {direction}")
rc = subprocess.run([
    sys.executable, "-m", "inkalign.cli", "sweep", "roi.zarr", ckpt,
    "--um-per-px", "9.362", "--outdir", "sweep_out", "--infer-template", TEMPLATE,
]).returncode
assert rc == 0, "sweep failed — scroll up for the infer error"

display(Image("sweep_out/sweep_curve.png"))
import json; print(json.dumps(json.load(open("sweep_out/sweep_summary.json"))["direction"], indent=2))

In [ ]:
# 5. Bundle everything for download (evidence for the README / submission)
!zip -qr inkalign_results.zip profile_out sweep_out
from google.colab import files
files.download("inkalign_results.zip")